# A Simple Compiler for a Fragment of  `C`

This file shows how a simple compiler for a fragment of the programming language `C` can be implemented using the F# port of `Larkita`.

First, we describe the grammar of our language using EBNF-notation, since this enables us to give a very concise presentation of the grammar.
Below, the token 'NUMBER' stands for natural numbers and 'ID' represents identifiers.
```
program: function+
       
function: 'int' ID '(' ('int', ID (',' 'int' ID)*)?  ')' '{' decl* stmnt* '}' 
    
decl: 'int' ID ';'  
    
stmnt: '{' stmnt* '}'
     | ID '=' expr ';' 
     | 'if' '(' boolExpr ')' stmnt 
     | 'if' '(' boolExpr ')' stmnt 'else' stmnt 
     | 'while' '(' boolExpr ')' stmnt 
     | 'return' expr ';'
     | expr ';'             
     
boolExpr: expr ('==' | '!=' | '<=' | '>=' | '<' | '>')  expr 
        | '!' boolExpr             
        | boolExpr ('&&' | '||') boolExpr
        | '(' boolExpr ')'
     
expr: expr ('+' | '-' | '*' | '/' | '%') expr 
    | '(' expr ')' 
    | NUMBER
    | ID ('(' (expr (',' expr)*)? ')')?  
```
The `Larkita` grammar below stratifies operator precedence; `Larkita` resolves the dangling `else` in favour of shifting. It additionally implements the bundled exercises: C-style `for`, range `for`, unary minus and the ternary conditional. Both loops are translated into assignments and `while` loops. 

## Specification of the Scanner

In [ ]:
// Nested tuples, the representation of abstract syntax trees; loaded once per session.
#load "../Support/Nested.fsx"

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#load "../Support/Larkita/Larkita.fsx"
open System
open System.IO
open Larkita
open Nested

Unlike `Ply`, which separates the scanner (`ply.lex`) from the parser (`ply.yacc`), `Larkita` reads a *single* grammar
that specifies both the *tokens* and the *grammar rules*:
- The names of *terminals*, i.e. of the tokens, are written in `UPPER_CASE`.  A terminal is defined either by a
  string enclosed in double quotes or by a regular expression enclosed in forward slashes.
- The names of *syntactical variables* are written in `lower_case`.

We store this grammar in small files in the directory `Grammar`; most of them describe the language of the lecture, while `statement-fsharp.lark` and `expr-fsharp.lark` describe the extensions of the F# edition, which are displayed and explained one after the
other.  The files of this section describe the scanner and are joined into the string `terminal_definitions`, while
the files of the next section contain the grammar rules and are joined into the string `syntax`.  At the end, both
strings are concatenated.

The parser returns the abstract syntax tree as a nested tuple of the type `nested` (see `../Support/Nested.fsx`).  The
code generator below works directly on these nested tuples.  The helper functions `str` and `tup` build leaves and
tuples, e.g. `tup [ str "+"; str "x"; str "y" ]` is the tuple `("+", "x", "y")`, and `number n` is the tuple
`("Number", "n")` that represents the number `n`.

In [ ]:
let str (s: string) : nested = Leaf (Str s)
let tup (components: nested list) : nested = Node components
let number (n: int) : nested = tup [ str "Number"; str (string n) ]

The token `NUMBER` specifies a natural number.  Note that the regular expression is enclosed in forward slashes.

In [ ]:
printfn "%s" (File.ReadAllText "Grammar/number.lark")

Next, we define the tokens for those operator symbols that consist of more than one character.  Strictly
speaking, these definitions are not necessary:  In `Larkita`, a string that occurs literally inside a grammar rule,
for example `"=="`, is turned into an *anonymous terminal* automatically.  However, an anonymous terminal that is
built from a symbol like `"=="` receives a generated name of the form `__ANON_1`, and such a name is of little help
when we inspect the token stream or when we read an error message.  If a terminal with the same definition has been
declared explicitly, `Larkita` uses this terminal instead of creating an anonymous one.  Therefore, the declarations
below only serve to give these tokens readable names.  `DOTDOT` is the range operator `..` of the `for` loop
`for (n = 2 .. 100)`.

Note that a terminal can also be defined by a string.  In this case no escaping is needed:  in contrast to the
`Ply` version, where the token `t_OR` had to be defined by the regular expression `r'\|\|'`, we can simply write
`"||"`.  Furthermore, `Larkita` sorts string terminals by their length, so `"!="` is preferred over `"!"` and `"<="` is
preferred over `"<"`.

The operators that consist of a single character need no declaration at all:  the list
```
literals = ['+', '-', '*', '/', '%', '(', ')', '{', '}', ';', '=', '<', '>', '!', ',']
```
that was needed for `Ply` has no counterpart in `Larkita`, since `Larkita` names the corresponding anonymous terminals
`PLUS`, `MINUS`, `STAR`, `LPAR`, `RPAR`, and so on.

In [ ]:
printfn "%s" (File.ReadAllText "Grammar/operators-for.lark")

Our version of `C` allows both *single-line comments* and *multi-line comments*.
- The regular expression `/\*(.|\n)*?\*/` recognizes multi-line comments.
  Multi-line comments start with the string `/*` and end with the string `*/`.
  Note the use of the *non-greedy* quantor `*?`.  If we have code like
  ```
  /* blah */ a = 1; /* blub */
  ```
  the greedy quantor would recognize the whole line as one comment. 
- The regular expression `//.*` recognizes single-line comments.
  A single line comment starts with the string `//` and extends to the end of the line.

A terminal can be defined as the alternative of several regular expressions.  As the forward slash is the
delimiter of a regular expression in `Larkita`, every occurrence of the character `/` inside a regular expression has
to be escaped as `\/`.

The directive `%ignore` tells the scanner to discard all tokens of the given type, i.e. comments never show up in
the token stream.  This is the `Larkita` equivalent of the function `t_COMMENT` that returned no token.

In [ ]:
printfn "%s" (File.ReadAllText "Grammar/comment.lark")

The keywords `'int'`, `'if'`, `'else'`, `'while'`, `'for'`, and `'return'` are syntactically identical to identifiers.
In the `Ply` version of this notebook this problem had been solved via the dictionary
```
Keywords = { 'int'   : 'INT', 
             'if'    : 'IF',
             'else'  : 'ELSE', 
             'while' : 'WHILE', 
             'return': 'RETURN'
           }
```
and the function `t_ID` that looked up every identifier in this dictionary.

`Larkita` performs this test automatically:  If a string terminal like `"if"` is also matched by the regular expression
of another terminal, in our case by the regular expression defining `NAME`, then `Larkita` first scans a `NAME` and
afterwards checks whether the resulting string is one of the keywords.  If it is, the type of the token is changed
accordingly.  This mechanism is known as the *unless* mechanism.  Because of this, the string `iffy` is scanned as a
single `NAME` and not as the keyword `if` followed by the identifier `fy`.

Therefore, all we have to specify is the terminal `NAME`.  The keywords themselves are written as strings inside the
grammar rules of the next section.

In [ ]:
printfn "%s" (File.ReadAllText "Grammar/name.lark")

White space, i.e. *space characters*, *tabulators*, *carriage returns*, and *newlines* are ignored.
In `Ply` we had needed two declarations for this:  the string `t_ignore` for the first three characters and the
function `t_newline`, which additionally had to increment the line number of the scanner.  In `Larkita`, a single
`%ignore` directive suffices, since `Larkita` keeps track of line numbers and column numbers on its own:  every token
`t` provides the fields `t.line`, `t.column`, `t.end_line`, and `t.end_column`.  Hence there is no need for a
function like `find_column` either.

In [ ]:
printfn "%s" (File.ReadAllText "Grammar/whitespace.lark")

The function `read_grammar` joins the given files of the directory `Grammar`.  The files of this section form the
complete specification of our scanner.

In [ ]:
let readGrammar (names: string list) =
    names |> List.map (fun name -> File.ReadAllText(Path.Combine(__SOURCE_DIRECTORY__, "Grammar", name + ".lark"))) |> String.concat "\n"

let terminalDefinitions = readGrammar [ "number"; "operators-for"; "comment"; "name"; "whitespace" ]

## Specification of the Parser

`Larkita` supports *EBNF* directly, so we can use the quantifiers `*`, `+`, and `?` as well as parentheses for
grouping.  There are two things that we have to take care of:
1. The expression grammar is ambiguous.  Since `Larkita` supports no precedence declarations, we encode the operator
   precedences into the grammar by introducing one syntactical variable per precedence level.
2. The grammar determines the abstract syntax tree that the parser returns:  The string literals of an alternative
   form the *head* of the tuple that is built for this alternative, while all other symbols become its children.
   *Aliases* `-> name` replace the head.

A program is a non-empty list of function definitions.  The alias `-> "."` makes the parser return a program as a
tuple of the form `(".", function, ...)`, while the alias `-> "fct"` yields function definitions of the form
`("fct", name, parameters, declarations, statements)`:  The string literals `"int"`, `"("`, `")"`, `"{"`, and `"}"`
would only contribute to the head of the tuple, but the alias replaces this head.

The directive `%ignore: "(", ")", "{", "}", ";"` (note the colon) removes the parentheses, the braces, and the
semicolon from the heads of *all* tuples, since these literals carry no information.

In [ ]:
printfn "%s" (File.ReadAllText "Grammar/program.lark")

The syntactical variables `parameters`, `declarations`, and `statements` describe lists.  In `Ply` these lists
had to be defined via recursion.  Using <span style="font-variant: small-caps;">Ebnf</span>, one rule per list
suffices.  The children of a repetition are spliced into the tuple of the rule.  Hence the alias `-> "."` yields tuples of
the form `(".", name, ...)` and `(".", statement, ...)`:  Note that the tuples of `parameters` and `declarations` contain
only the tokens of type `NAME`, since all other symbols of these rules are string literals, which only contribute to
the head.

In [ ]:
printfn "%s" (File.ReadAllText "Grammar/lists.lark")

The rule `statement` extends the statements of the lecture by two `for` loops:  the C-style loop
`for (f = 2; f < n; f = f + 1)` yields `("for", init, test, update, body)` and the range loop `for (n = 2 .. 100)` yields
`("for-range", ("NAME", n), lower, upper, body)`.  Both are translated into assignments and `while` loops by the code generator.  An expression statement gets the alias `-> "expr"`, so that it is not confused with the
expression itself.

The alternatives for `if` and `if`-`else` cause the well-known
[dangling else problem](https://en.wikipedia.org/wiki/Dangling_else):  `Larkita` resolves this *shift/reduce
conflict* in favour of shifting, so the keyword `else` is attached to the innermost `if` that does not yet have an
`else` part.

In [ ]:
printfn "%s" (File.ReadAllText "Grammar/statement-fsharp.lark")

Arithmetic, Boolean and ternary precedence are expressed with layered grammar variables.  The order from weakest to
strongest is the ternary operator `?:`, `||`, `&&`, Boolean `!`, the comparisons, `+` and `-`, `*`, `/` and `%`, and
finally the unary minus.  The operators are the heads of the tuples, e.g. `x < y` yields `("<", x, y)`, while the
ternary operator yields `(("?", ":"), test, yes, no)` and the unary minus `("neg", x)`.  As `!` binds weaker than the
comparisons, `!x < y` means `!(x < y)`.

In [ ]:
printfn "%s" (File.ReadAllText "Grammar/expr-fsharp.lark")

The complete grammar is the concatenation of the grammar rules and the terminal definitions.

In [ ]:
let syntax = readGrammar [ "program"; "lists"; "statement-fsharp"; "expr-fsharp" ]
let grammar = syntax + "\n" + terminalDefinitions

In [ ]:
printfn "%s" grammar

Now we can create the parser.  `Larkita` always generates a *canonical LR(1)* parser.  The argument
`start = "program"` declares `program` to be the *start variable* of our grammar.  It could be omitted, since the
start variable defaults to the first rule of the grammar.

`Larkita` writes a description of all states of the LR(1) automaton into the file given by the argument
`outFile`.  This is the `Larkita` equivalent of the file `parser.out` that had been produced by `Ply`.  If there are
shift/reduce conflicts, `Larkita` prints a warning.  Reduce/reduce conflicts are errors.

In [ ]:
let cParser = Larkita.Create(grammar, start = "program", outFile = Path.Combine(__SOURCE_DIRECTORY__, "compiler.out"))

The only conflict that is reported is the shift/reduce conflict that results from the *dangling else
ambiguity*.  As discussed above, `Larkita` resolves this conflict in favour of shifting and hence in the way that we
want.  All other conflicts have been eliminated by stratifying the grammar.  The following cell prints the section
`Conflicts` of the file `compiler.out`.

In [ ]:
let report = File.ReadAllText cParser.OutFile
printfn "%s" (report.Substring(report.IndexOf "Conflicts\n", report.IndexOf "States\n" - report.IndexOf "Conflicts\n"))
checkNotebook (cParser.Conflicts.Length = 1)

## Construction of the Abstract Syntax Tree

The function `simplifyLeaves` simplifies the leaves of the nested tuple that `Larkita` returns:  a variable
`("NAME", name)` is replaced by the string `name` and a number `("NUMBER", digits)` is replaced by the tuple
`("Number", digits)`.  Then a string is always a variable.  The module `Frontend` combines parsing and
simplification; `Frontend.scan` runs only the scanner.

In [ ]:
let unexpected (t: nested) = failwithf "Unexpected tuple in the abstract syntax tree: %s" (toString t)

let rec simplifyLeaves (ast: nested) : nested =
    match ast with
    | Node [ Leaf (Str "NAME"); name ] -> name
    | Node [ Leaf (Str "NUMBER"); digits ] -> tup [ str "Number"; digits ]
    | Node (head :: children) -> Node (head :: List.map simplifyLeaves children)
    | _ -> ast

/// leafName t: the name of a variable, i.e. the string of the leaf t
let leafName (t: nested) =
    match t with
    | Leaf (Str name) -> name
    | _ -> unexpected t

/// functionParts f: the name, the parameters, the local variables, and the statements of the function f
let functionParts (definition: nested) =
    match definition with
    | Node [ Leaf (Str "fct"); Leaf (Str name); Node (_ :: parameters); Node (_ :: variables); Node (_ :: body) ] ->
        name, List.map leafName parameters, List.map leafName variables, body
    | _ -> unexpected definition

/// functions p: the function definitions of the program p
let functions (program: nested) =
    match program with
    | Node (Leaf (Str ".") :: definitions) -> definitions
    | _ -> unexpected program

module Frontend =
    /// the nested tuple of a program; a syntax error raises UnexpectedInput with its line and column
    let parse source = simplifyLeaves (parse cParser source)
    /// the tokens of a program with their types, texts, lines and columns
    let scan source = lex cParser source |> List.map (fun t -> t.Type, t.Value, t.Line, t.Column)

## Testing the Scanner

`Frontend.scan` runs the scanner without invoking the parser.  It returns the list of tokens together with their
types, texts, lines and columns.  Note that white space and comments do not show up because of the `%ignore`
directives, and that the keywords have been retyped from `NAME` to `INT`, `IF`, `ELSE`, `WHILE`, `FOR`, and `RETURN`.

In [ ]:
let position source = Frontend.scan source |> List.map (fun (_, _, line, column) -> line, column)

In [ ]:
let scanner = Frontend.scan

In [ ]:
let testScanner fileName =
    let source = File.ReadAllText fileName
    printfn "%s" source
    scanner source

In [ ]:
File.ReadAllText("Examples/MySum.c") |> printfn "%s"

In [ ]:
for token in testScanner "Examples/MySum.c" do printfn "%A" token

## Parsing

In [ ]:
let parser = Frontend.parse

The stratified grammar leaves only the dangling-else conflict, which is resolved in favour of shifting.

The nested tuple is printed in tuple notation and converted to Graphviz DOT by the recursive function below.  The
label of a node is the first component of its tuple.  Open the saved DOT file in Graphviz, or render it to SVG with
`dot -Tsvg`.

In [ ]:
let astToDot (program: nested) =
    let output = System.Text.StringBuilder("digraph AST {\n  node [shape=box];\n")
    let mutable nextId = 0
    let node (title: string) (children: string list) =
        let id = "n" + string nextId
        nextId <- nextId + 1
        let escaped = title.Replace("\\", "\\\\").Replace("\"", "\\\"")
        output.AppendLine(sprintf "  %s [label=\"%s\"];" id escaped) |> ignore
        for child in children do output.AppendLine(sprintf "  %s -> %s;" id child) |> ignore
        id
    let rec visit (t: nested) =
        match t with
        | Leaf (Str s) -> node s []
        | Leaf v -> node (valueToString v) []
        | Node (Leaf (Str title) :: children) -> node title (List.map visit children)
        | Node (head :: children) -> node (toString head) (List.map visit children)
        | Node [] -> node "()" []
    visit program |> ignore
    output.AppendLine("}").ToString()

`parse fileName` reads a C source file and returns its abstract syntax tree. The frontend starts a fresh scanner buffer for each call and reports source positions on errors.

In [ ]:
let parse fileName =
    let ast = parser (File.ReadAllText fileName)
    printfn "%s" (toString ast)
    ast

In [ ]:
File.ReadAllLines("Examples/MySum.c") |> Array.iteri (fun i line -> printfn "%3d %s" (i+1) line)

In [ ]:
let sumAst = parse "Examples/MySum.c"
let dot = astToDot sumAst
File.WriteAllText("Examples/MySum.dot",dot)
printfn "%s" dot
// Optional rendering: dot -Tsvg Examples/MySum.dot -o Examples/MySum.svg

The function `indent` is used to indent the generated assembler commands by preceding them with 8 space characters. 

In [ ]:
let indent command = "        " + command
let label name = "    " + name + ":"
let createLabels () =
    let mutable count = 0
    fun () -> count <- count + 1; "l" + string count

type CompilationContext = {
    Symbols: Map<string,int>
    Arities: Map<string,int>
    ClassName: string
    NewLabel: unit -> string
    ReturnsVoid: bool
}

`compileExpr`, `compileBool` and `compileStatement` receive a `CompilationContext` and a nested tuple. The context holds local-variable slots, function arities, class name and a label factory. Each returns `(instructions, maximumStack)`, so the JVM stack limit is computed while generating code. Arithmetic operands contribute different temporary stack heights; branch arms meet at equal heights.

In [ ]:
let rec compileExpr context (expression: nested) : string list * int =
    let command text = indent text
    let binary instruction lhs rhs =
        let left, leftSize = compileExpr context lhs
        let right, rightSize = compileExpr context rhs
        left @ right @ [command instruction], max leftSize (1 + rightSize)
    match expression with
    | Node [ Leaf (Str "Number"); Leaf (Str digits) ] -> [command ("ldc " + string (int digits))], 1
    | Leaf (Str name) ->
        match Map.tryFind name context.Symbols with
        | Some slot -> [command ("iload " + string slot)], 1
        | None -> failwithf "Undeclared variable: %s" name
    | Node [ Leaf (Str "neg"); value ] -> compileExpr context (tup [ str "-"; number 0; value ])
    | Node [ Leaf (Str (("+" | "-" | "*" | "/" | "%") as op)); lhs; rhs ] ->
        let instruction = Map.ofList ["+","iadd"; "-","isub"; "*","imul"; "/","idiv"; "%","irem"]
        binary instruction.[op] lhs rhs
    | Node [ Leaf (Str (("==" | "!=" | "<=" | ">=" | "<" | ">") as op)); lhs; rhs ] ->
        let instructions = Map.ofList ["==","if_icmpeq"; "!=","if_icmpne"; "<=","if_icmple"; ">=","if_icmpge"; "<","if_icmplt"; ">","if_icmpgt"]
        let left, leftSize = compileExpr context lhs
        let right, rightSize = compileExpr context rhs
        let yes, doneLabel = context.NewLabel(), context.NewLabel()
        left @ right @ [command (instructions.[op] + " " + yes); command "iconst_0"; command ("goto " + doneLabel); label yes; command "iconst_1"; label doneLabel], max leftSize (1 + rightSize)
    | Node [ Leaf (Str (("&&" | "||") as op)); lhs; rhs ] ->
        // Short circuit each operand, leaving exactly one normalized Boolean on the stack.
        let left, leftSize = compileExpr context lhs
        let right, rightSize = compileExpr context rhs
        let decisive, doneLabel = context.NewLabel(), context.NewLabel()
        let branch, defaultValue, decisiveValue = if op = "&&" then "ifeq", "iconst_1", "iconst_0" else "ifne", "iconst_0", "iconst_1"
        left @ [command (branch + " " + decisive)] @ right @ [command (branch + " " + decisive); command defaultValue; command ("goto " + doneLabel); label decisive; command decisiveValue; label doneLabel], max 1 (max leftSize rightSize)
    | Node [ Leaf (Str "!"); value ] -> compileExpr context (tup [ tup [ str "?"; str ":" ]; value; number 0; number 1 ])
    | Node [ Node [ Leaf (Str "?"); Leaf (Str ":") ]; condition; yes; no ] ->
        let test, testSize = compileExpr context condition
        let left, leftSize = compileExpr context yes
        let right, rightSize = compileExpr context no
        let otherwise, doneLabel = context.NewLabel(), context.NewLabel()
        test @ [command ("ifeq " + otherwise)] @ left @ [command ("goto " + doneLabel); label otherwise] @ right @ [label doneLabel], max testSize (max leftSize rightSize)
    | Node (Leaf (Str "call") :: Leaf (Str "println") :: _) -> failwith "println returns void and is only valid as a statement"
    | Node (Leaf (Str "call") :: Leaf (Str name) :: arguments) ->
        match Map.tryFind name context.Arities with
        | Some arity when arity = arguments.Length -> ()
        | Some arity -> failwithf "%s expects %d arguments, got %d" name arity arguments.Length
        | None -> failwithf "Unknown function: %s" name
        let instructions, size, _ =
            arguments |> List.fold (fun (commands,size,count) arg ->
                let generated, argSize = compileExpr context arg
                commands @ generated, max size (count + argSize), count + 1) ([],0,0)
        instructions @ [command (sprintf "invokestatic %s/%s(%s)I" context.ClassName name (String.replicate arguments.Length "I"))], max 1 size
    | _ -> failwithf "Unsupported expression: %s" (toString expression)

The following is a test of the function `compile_expr`.

In [ ]:
let exampleContext = { Symbols=Map.ofList ["x",0; "y",1]; Arities=Map.ofList ["sum",1]; ClassName="Sum"; NewLabel=createLabels (); ReturnsVoid=false }
compileExpr exampleContext (tup [ str "call"; str "sum"; tup [ str "+"; str "x"; str "y" ] ])

`createLabels` returns a label factory with a private counter. Each method gets its own factory, preventing notebook execution history from changing generated labels.

In [ ]:
let newLabel = createLabels ()
newLabel (), newLabel ()

`compileExpr`, `compileBool` and `compileStatement` receive a `CompilationContext` and a nested tuple. The context holds local-variable slots, function arities, class name and a label factory. Each returns `(instructions, maximumStack)`, so the JVM stack limit is computed while generating code. Arithmetic operands contribute different temporary stack heights; branch arms meet at equal heights.

In [ ]:
// Boolean expressions are normalized by the same expression compiler.
let compileBool context expression = compileExpr context expression

Below is a test for the function `compile_bool`.

In [ ]:
compileBool exampleContext (tup [ str "=="; str "x"; number 0 ])

`compileExpr`, `compileBool` and `compileStatement` receive a `CompilationContext` and a nested tuple. The context holds local-variable slots, function arities, class name and a label factory. Each returns `(instructions, maximumStack)`, so the JVM stack limit is computed while generating code. Arithmetic operands contribute different temporary stack heights; branch arms meet at equal heights.

In [ ]:
let rec compileStatement context (statement: nested) : string list * int =
    let expr = compileExpr context
    let stmnt = compileStatement context
    match statement with
    | Node [ Leaf (Str "="); Leaf (Str name); value ] ->
        let commands,size = expr value
        match Map.tryFind name context.Symbols with
        | Some slot -> commands @ [indent ("istore " + string slot)], size
        | None -> failwithf "Assignment to undeclared variable: %s" name
    | Node [ Leaf (Str "if"); condition; yes ] -> stmnt (tup [ str "if-else"; condition; yes; tup [ str "." ] ])
    | Node [ Leaf (Str "if-else"); condition; yes; no ] ->
        let test,testSize = expr condition
        let yesCommands,yesSize = stmnt yes
        let noCommands,noSize = stmnt no
        let otherwise,doneLabel = context.NewLabel(),context.NewLabel()
        test @ [indent ("ifeq " + otherwise)] @ yesCommands @ [indent ("goto " + doneLabel); label otherwise] @ noCommands @ [label doneLabel], max testSize (max yesSize noSize)
    | Node [ Leaf (Str "while"); condition; body ] ->
        let test,testSize = expr condition
        let bodyCommands,bodySize = stmnt body
        let loop,doneLabel = context.NewLabel(),context.NewLabel()
        [label loop] @ test @ [indent ("ifeq " + doneLabel)] @ bodyCommands @ [indent ("goto " + loop); label doneLabel], max testSize bodySize
    // Both for loops are translated into assignments and while loops.
    | Node [ Leaf (Str "for"); initial; test; update; body ] ->
        stmnt (tup [ str "."; initial; tup [ str "while"; test; tup [ str "."; body; update ] ] ])
    | Node [ Leaf (Str "for-range"); variable; lower; upper; body ] ->
        let increment = tup [ str "="; variable; tup [ str "+"; variable; number 1 ] ]
        stmnt (tup [ str "."; tup [ str "="; variable; lower ]
                     tup [ str "while"; tup [ str "<="; variable; upper ]; tup [ str "."; body; increment ] ] ])
    | Node [ Leaf (Str "return"); value ] ->
        let commands,size = expr value
        commands @ (if context.ReturnsVoid then [indent "pop"; indent "return"] else [indent "ireturn"]), size
    | Node (Leaf (Str ".") :: statements) ->
        statements |> List.fold (fun (commands,size) statement -> let next,nextSize = stmnt statement in commands @ next, max size nextSize) ([],0)
    | Node [ Leaf (Str "expr"); Node [ Leaf (Str "call"); Leaf (Str "println"); value ] ] ->
        let commands,size = expr value
        [indent "getstatic java/lang/System/out Ljava/io/PrintStream;"] @ commands @ [indent "invokevirtual java/io/PrintStream/println(I)V"], 1 + size
    | Node [ Leaf (Str "expr"); Node (Leaf (Str "call") :: Leaf (Str "println") :: _) ] -> failwith "println requires exactly one integer argument"
    | Node [ Leaf (Str "expr"); value ] ->
        let commands,size = expr value
        commands @ [indent "pop"], size // Discard unused values to keep loop stack heights consistent.
    | _ -> failwithf "Unsupported statement: %s" (toString statement)

In [ ]:
compileStatement exampleContext (tup [ str "if"; tup [ str "=="; str "x"; str "y" ]; tup [ str "="; str "x"; number 42 ] ])

In [ ]:
compileStatement exampleContext (tup [ str "if-else"; tup [ str "<"; str "x"; str "y" ]; tup [ str "="; str "x"; str "y" ]; tup [ str "="; str "y"; str "x" ] ])

In [ ]:
compileStatement exampleContext (tup [ str "while"; tup [ str "<"; str "x"; str "y" ]; tup [ str "="; str "x"; tup [ str "+"; str "x"; number 1 ] ] ])

In [ ]:
compileStatement exampleContext (tup [ str "."; tup [ str "="; str "x"; str "y" ]; tup [ str "."; tup [ str "="; str "x"; number 1 ]; tup [ str "="; str "y"; str "x" ] ] ])

In [ ]:
let compileFunction className arities (definition: nested) =
    let name, parameters, variables, body = functionParts definition
    let isMain = name = "main"
    if isMain && not parameters.IsEmpty then failwith "main must have no C parameters"
    let names = parameters @ variables
    if (Set.ofList names).Count <> names.Length then failwithf "Duplicate local or parameter in %s" name
    // JVM main slot 0 contains String[] args, so Integer C locals start at slot 1.
    let firstSlot = if isMain then 1 else 0
    let symbols = names |> List.mapi (fun i name -> name,i+firstSlot) |> Map.ofList
    let context = {Symbols=symbols; Arities=arities; ClassName=className; NewLabel=createLabels (); ReturnsVoid=isMain}
    let commands,size = compileStatement context (tup (str "." :: body))
    let initialize = variables |> List.collect (fun name -> [indent "iconst_0"; indent ("istore " + string symbols.[name])])
    let signature = if isMain then ".method public static main([Ljava/lang/String;)V" else sprintf ".method public static %s(%s)I" name (String.replicate parameters.Length "I")
    let fallback = if isMain then [indent "return"] else [indent "iconst_0"; indent "ireturn"]
    [signature; sprintf ".limit locals %d" (firstSlot + names.Length); sprintf ".limit stack %d" (max 1 size)] @ initialize @ commands @ fallback @ [".end method"]

In [ ]:
let sumFunction = tup [ str "fct"; str "sum"; tup [ str "."; str "x" ]; tup [ str "."; str "y"; str "z" ]; tup [ str "."; tup [ str "return"; str "x" ] ] ]
compileFunction "Sum" (Map.ofList ["sum",1]) sumFunction

In [ ]:
let file = "Examples/Primes.c"
printfn "%s" (Path.GetDirectoryName file)
printfn "%s" (Path.GetFileName file)
printfn "%s" (Path.Combine("abc","xyz.c"))

`compileProgram fileName` parses a `.c` file, checks function/local declarations and call arities, and writes sibling `.jas` assembly. JVM `main` reserves slot 0 for its `String[]` argument. Expression statements discard unused values and `println` is explicitly void.

In [ ]:
let compileProgram (fileName: string) =
    if Path.GetExtension(fileName) <> ".c" then invalidArg "fileName" "Expected a .c source file"
    let className = Path.GetFileNameWithoutExtension fileName
    if not (System.Text.RegularExpressions.Regex.IsMatch(className,@"^[A-Za-z][A-Za-z0-9_]*$")) then invalidArg "fileName" "Class name must be a JVM identifier"
    let functions = functions (Frontend.parse (File.ReadAllText fileName))
    let parts = functions |> List.map functionParts
    let functionNames = parts |> List.map (fun (name, _, _, _) -> name)
    if (Set.ofList functionNames).Count <> functions.Length then failwith "Duplicate function definition"
    if not (List.contains "main" functionNames) then failwith "Program needs a main function"
    let arities =
        parts
        |> List.filter (fun (name, _, _, _) -> name <> "main")
        |> List.map (fun (name, parameters, _, _) -> name, parameters.Length)
        |> Map.ofList
    let header = [".class public " + className; ".super java/lang/Object"; ""; ".method public <init>()V"; ".limit stack 1"; ".limit locals 1"; indent "aload_0"; indent "invokespecial java/lang/Object/<init>()V"; indent "return"; ".end method"; ""]
    let methods = functions |> List.collect (fun f -> compileFunction className arities f @ [""])
    let output = Path.ChangeExtension(fileName,".jas")
    File.WriteAllLines(output,header @ methods)
    output

The file `Primes.c` is a simple `C` program that computes the prime numbers in a naive way.

In [ ]:
Directory.GetFiles("Examples","*.c") |> Array.iter (printfn "%s")

In [ ]:
File.ReadAllText("Examples/Primes.c") |> printfn "%s"

In [ ]:
let primeAssembler = compileProgram "Examples/Primes.c"

In [ ]:
Directory.GetFiles("Examples","*.jas") |> Array.iter (printfn "%s")

In [ ]:
File.ReadAllText(primeAssembler) |> printfn "%s"

Optionally assemble the generated `.jas` file with Jasmin after installing a JDK and Jasmin. The notebook defaults to deterministic F# compilation and does not download or install external executables.

In [ ]:
open System.Diagnostics
let runTool (executable: string) (arguments: string list) (workingDirectory: string) =
    let info = ProcessStartInfo(executable)
    info.WorkingDirectory <- workingDirectory
    info.UseShellExecute <- false
    info.RedirectStandardOutput <- true
    info.RedirectStandardError <- true
    for argument in arguments do info.ArgumentList.Add argument
    use childProcess = Process.Start(info)
    let output = childProcess.StandardOutput.ReadToEnd()
    let errors = childProcess.StandardError.ReadToEnd()
    childProcess.WaitForExit()
    if childProcess.ExitCode <> 0 then failwithf "%s failed: %s" executable errors
    output
// Optional external execution after installing a JDK and Jasmin:
// runTool "jasmin" ["Primes.jas"] "Examples" |> printfn "%s"

After assembly, run the class with Java. This step requires the external tools; successful F# execution alone does not prove JVM execution.

In [ ]:
// Generated source files can be inspected before optional JVM assembly.
for file in Directory.GetFiles("Examples","*.c") do
    let assembler = compileProgram file
    let text = File.ReadAllText assembler
    checkNotebook (text.Contains(".limit stack") && text.Contains(".limit locals") && text.Contains("main([Ljava/lang/String;)V"))
    printfn "Compiled %s" file

In [ ]:
// Optional after assembly: runTool "java" ["-cp"; "."; "Primes"] "Examples" |> printfn "%s"

## Verification

These checks cover arithmetic and Boolean precedence, dangling else, malformed input, symbol boundaries and JVM stack balance. The optional JVM script described in README.md assembles and executes all six bundled examples with bytecode verification enabled. It also checks that `!x < y` and `!(x < y)` both produce zero for `x = 1`, `y = 2`, and that `&&`/`||` skip a right operand containing division by zero or a function call with observable output.


In [ ]:
let expectFailure action =
    let mutable failed = false
    try action () with _ -> failed <- true
    checkNotebook failed
expectFailure (fun () -> Frontend.parse "int main() { @ }" |> ignore)
expectFailure (fun () -> Frontend.parse "int main() { println(1);" |> ignore)
/// the statements of the first function of a program
let firstBody program =
    let _, _, _, body = functionParts (List.head (functions program))
    body
let println value = tup [ str "expr"; tup [ str "call"; str "println"; value ] ]
let precedenceAst = Frontend.parse "int main() { println(1 + 2 * 3); }"
checkNotebook (firstBody precedenceAst = [ println (tup [ str "+"; number 1; tup [ str "*"; number 2; number 3 ] ]) ])
// The lecture places Boolean negation below comparison operators.
let negatedComparison = tup [ str "!"; tup [ str "<"; str "x"; str "y" ] ]
for source in ["!x < y"; "!(x < y)"] do
    let ast = Frontend.parse ("int main() { println(" + source + "); }")
    if firstBody ast <> [ println negatedComparison ] then failwithf "Incorrect Boolean negation precedence: %s" source
let danglingElse = Frontend.parse "int main() { if (1) if (0) println(1); else println(2); }"
match firstBody danglingElse with
| [ Node [ Leaf (Str "if"); _; Node [ Leaf (Str "if-else"); _; _; _ ] ] ] -> ()
| _ -> failwith "Else must attach to the inner if"
expectFailure (fun () -> compileExpr exampleContext (str "undeclared") |> ignore)
expectFailure (fun () -> compileExpr exampleContext (tup [ str "call"; str "sum" ]) |> ignore)
let unusedValue,_ = compileStatement exampleContext (tup [ str "expr"; number 42 ])
checkNotebook (unusedValue |> List.last = indent "pop")
let booleanCommands,booleanStack = compileBool exampleContext (tup [ str "!"; tup [ str "=="; str "x"; number 0 ] ])
checkNotebook (booleanStack >= 2 && booleanCommands |> List.exists (fun line -> line.Contains("ifeq")))
printfn "Arithmetic/Boolean precedence, diagnostics, arity, unused values and Boolean code generation passed."
